In [ ]:
#| default_exp train_flow

# Flow Matching Generative Model

Trains a flow matching model on pre-encoded (optionally PCA-reduced) embeddings.
Source distribution is N(0,I); target is the embedding distribution.
Uses RK4 integration and optional time warping at inference.

In [ ]:
#| export
import gc
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
from midi_rae.data import EmbeddingDataset

In [ ]:
#| export
class VelocityNet(nn.Module):
    """MLP velocity field for flow matching.  Input: [x, (x_self_cond,) t_emb], output: dx/dt.
    Hidden layers use residual (skip) connections.
    t_dim: sinusoidal time embedding dim (replaces bare scalar t).
    self_condition: if True, also accepts x_self_cond (predicted x1 from prior pass); zeros when absent."""
    def __init__(self, input_dim, h_dim=256, n_layers=3, self_condition=False, t_dim=64):
        super().__init__()
        self.self_condition = self_condition
        self.t_dim = t_dim
        net_in = input_dim * 2 + t_dim if self_condition else input_dim + t_dim
        self.fc_in  = nn.Linear(net_in, h_dim)
        self.hidden = nn.ModuleList([nn.Linear(h_dim, h_dim) for _ in range(n_layers - 1)])
        self.fc_out = nn.Linear(h_dim, input_dim)

    def forward(self, x, t, x_self_cond=None):
        if t.dim() > 1: t = t.squeeze(-1)
        elif t.dim() == 0: t = t.unsqueeze(0).expand(x.size(0))
        t_emb = sinusoidal_time_emb(t, self.t_dim)          # [B, t_dim]
        if self.self_condition:
            sc = x_self_cond if x_self_cond is not None else torch.zeros_like(x)
            inp = torch.cat([x, sc, t_emb], dim=1)
        else:
            inp = torch.cat([x, t_emb], dim=1)
        h = F.gelu(self.fc_in(inp))
        for layer in self.hidden:
            h = F.gelu(layer(h)) + h
        return self.fc_out(h)


In [ ]:
#| export
import math

def sinusoidal_time_emb(t, dim=64):
    """Sinusoidal time embedding (à la DDPM/DiT).  t: [B] or [B,1] → [B, dim].
    Gives the model a rich multi-frequency view of t instead of a bare scalar."""
    if t.dim() > 1: t = t.squeeze(-1)
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, dtype=torch.float32, device=t.device) / (half - 1))
    x = t.float().unsqueeze(1) * freqs.unsqueeze(0)   # [B, half]
    return torch.cat([x.sin(), x.cos()], dim=-1)       # [B, dim]


In [ ]:
#| export
class PerLevelFlowModel(nn.Module):
    """One VelocityNet per embedding level; each level's slice is routed to its own net.
    Has the same forward(x, t, x_self_cond=None) interface as VelocityNet.
    level_dims: list of ints, e.g. [20, 80, 320, 1280] from dataset.level_dims
    self_condition / t_dim: passed through to each VelocityNet.
    """
    def __init__(self, level_dims, h_dim=256, n_layers=4, self_condition=False, t_dim=64):
        super().__init__()
        self.self_condition = self_condition
        self.level_dims = level_dims
        self.nets = nn.ModuleList([VelocityNet(d, h_dim, n_layers, self_condition=self_condition, t_dim=t_dim)
                                   for d in level_dims])

    def forward(self, x, t, x_self_cond=None):
        outs, offset = [], 0
        for net, d in zip(self.nets, self.level_dims):
            sc_slice = x_self_cond[:, offset:offset+d] if x_self_cond is not None else None
            outs.append(net(x[:, offset:offset+d], t, sc_slice))
            offset += d
        return torch.cat(outs, dim=1)


In [ ]:
#| export
class CrossLevelFlowModel(nn.Module):
    """Flow model with cross-level attention for joint velocity prediction.
    Each level is projected to h_dim, t is embedded sinusoidally and added to every token,
    a small transformer cross-attends across all levels (so L3 attends to L0-L2 without
    cascading), per-level residual MLPs refine, per-level heads decode velocity.
    Sequence length = n_levels (typically 4) so attention cost is negligible.
    Uses norm_first=True (pre-LN) for training stability.
    self_condition: same 50%-dropout self-conditioning as VelocityNet.
    t_dim: sinusoidal time embedding dim, projected to h_dim and added to each level token.
    """
    def __init__(self, level_dims, h_dim=512, n_layers=4, n_attn_layers=2, n_heads=8,
                 self_condition=False, t_dim=64):
        super().__init__()
        self.self_condition = self_condition
        self.level_dims = level_dims
        self.t_dim = t_dim
        in_mul = 2 if self_condition else 1
        # Per-level input projections: [x_level, (x_sc)] → h_dim  (t added separately)
        self.level_in  = nn.ModuleList([nn.Linear(d * in_mul, h_dim) for d in level_dims])
        # Time embedding: sinusoidal t_dim → h_dim (added to every level token)
        self.t_proj = nn.Sequential(nn.Linear(t_dim, h_dim), nn.SiLU(), nn.Linear(h_dim, h_dim))
        # Cross-level transformer (seq_len = n_levels ≈ 4; norm_first=True for stability)
        enc_layer = nn.TransformerEncoderLayer(h_dim, n_heads, dim_feedforward=h_dim * 4,
                                               batch_first=True, dropout=0.0, norm_first=True)
        self.cross_attn = nn.TransformerEncoder(enc_layer, num_layers=n_attn_layers)
        # Per-level residual MLPs (operate in h_dim space)
        self.level_mlp = nn.ModuleList([
            nn.ModuleList([nn.Linear(h_dim, h_dim) for _ in range(n_layers - 1)])
            for _ in level_dims])
        # Per-level output heads → velocity
        self.level_out = nn.ModuleList([nn.Linear(h_dim, d) for d in level_dims])

    def forward(self, x, t, x_self_cond=None):
        if t.dim() > 1: t = t.squeeze(-1)
        elif t.dim() == 0: t = t.unsqueeze(0).expand(x.size(0))
        t_emb = self.t_proj(sinusoidal_time_emb(t, self.t_dim))   # [B, h_dim]
        # Build per-level tokens
        tokens, offset = [], 0
        for proj, d in zip(self.level_in, self.level_dims):
            xd = x[:, offset:offset+d]
            if self.self_condition:
                sc = x_self_cond[:, offset:offset+d] if x_self_cond is not None else torch.zeros_like(xd)
                inp = torch.cat([xd, sc], dim=1)
            else:
                inp = xd
            tokens.append(F.gelu(proj(inp)) + t_emb)   # add time additively
            offset += d
        # Cross-attend across levels: [B, n_levels, h_dim]
        tokens = self.cross_attn(torch.stack(tokens, dim=1))
        # Per-level residual MLP + output head
        outs = []
        for mlp_layers, out_proj, tok in zip(self.level_mlp, self.level_out, tokens.unbind(1)):
            h = tok
            for layer in mlp_layers:
                h = F.gelu(layer(h)) + h
            outs.append(out_proj(h))
        return torch.cat(outs, dim=1)


In [ ]:
#| export
def warp_time(t, s=0.5):
    """Parametric time warping (Scott H. Hawley, 'Flow With What You Know', ICLR 2025).
    s=1 → linear; s<1 → slower near middle; s=1.5 ≈ cosine schedule.
    Works on scalar, 1-D or 2-D tensors."""
    return 4*(1-s)*t**3 + 6*(s-1)*t**2 + (3-2*s)*t

In [ ]:
#| export
@torch.no_grad()
def rk4_step(model, y, t, dt):
    """4th-order Runge-Kutta step for the learned velocity field."""
    t_  = torch.full((y.size(0), 1), t, device=y.device, dtype=y.dtype)
    k1 = model(y,             t_)
    k2 = model(y + dt*k1/2,   t_ + dt/2)
    k3 = model(y + dt*k2/2,   t_ + dt/2)
    k4 = model(y + dt*k3,     t_ + dt)
    return y + (dt/6)*(k1 + 2*k2 + 2*k3 + k4)

@torch.no_grad()
def euler_step(model, y, t, dt):
    t_ = torch.full((y.size(0), 1), t, device=y.device, dtype=y.dtype)
    return y + model(y, t_) * dt

In [ ]:
#| export
def sample_source(shape, device='cpu', source_df=None, source_scales=None, level_dims=None):
    """Sample from source distribution with optional per-level Student-t and scaling.
    source_df: scalar df → Student-t for all dims; list → per-level (None/0 = Gaussian, float = Student-t)
    source_scales: list of per-level scale factors applied after sampling
    """
    if isinstance(source_df, (list, tuple)):
        # Per-level: each slice sampled independently
        assert level_dims is not None, "level_dims required for per-level source_df"
        batch = shape[:-1]
        y = torch.empty(*shape, device=device)
        offset = 0
        for df, d in zip(source_df, level_dims):
            sl = (*batch, d)
            if df:
                normal = torch.randn(*sl, device=device)
                gamma  = torch._standard_gamma(torch.full(sl, df/2, device=device)) / (df/2)
                y[..., offset:offset+d] = normal / gamma.sqrt()
            else:
                y[..., offset:offset+d] = torch.randn(*sl, device=device)
            offset += d
    elif source_df:
        # Scalar df → Student-t for all dims
        normal = torch.randn(*shape, device=device)
        gamma  = torch._standard_gamma(torch.full(shape, source_df/2, device=device)) / (source_df/2)
        y = normal / gamma.sqrt()
    else:
        y = torch.randn(*shape, device=device)
    if source_scales is not None and level_dims is not None:
        offset = 0
        for scale, d in zip(source_scales, level_dims):
            y[..., offset:offset+d] *= scale
            offset += d
    return y

In [ ]:
#| export
def ann_repair(source, target, n_projections=1, chunk_size=None):
    """Approximate nearest-neighbor re-pairing of source and target batches.

    Sorts both source and target by their projection onto random unit vectors
    and pairs by rank — equivalent to exact 1-D OT along that direction.
    Fully on-device (GPU-friendly), O(B log B) per projection.

    chunk_size: if set, processes the batch in chunks of this size and repairs
    independently within each chunk. Smaller chunks are faster but less optimal;
    default (None) processes the whole batch at once.

    With n_projections > 1, tries multiple random directions and keeps the
    pairing with the lowest total squared transport cost.

    Args:
        source:        (B, D) tensor on any device
        target:        (B, D) tensor on same device
        n_projections: number of random projections to try per chunk
        chunk_size:    chunk size for within-batch processing (None = full batch)

    Returns:
        (source_repaired, target_repaired): re-ordered so source[i] ↔ target[i]
        approximately minimises total squared transport cost.
    """
    B, D = source.shape
    C = B if (chunk_size is None or chunk_size >= B) else chunk_size

    s_out = torch.empty_like(source)
    t_out = torch.empty_like(target)
    for start in range(0, B, C):
        end  = min(start + C, B)
        s, t = source[start:end], target[start:end]
        best_s, best_t, best_cost = s, t, float('inf')
        for _ in range(n_projections):
            proj   = torch.randn(D, device=s.device, dtype=s.dtype)
            proj   = proj / proj.norm()
            s_rep  = s[(s @ proj).argsort()]
            t_rep  = t[(t @ proj).argsort()]
            cost   = (s_rep - t_rep).pow(2).sum().item()
            if cost < best_cost:
                best_cost = cost
                best_s, best_t = s_rep, t_rep
        s_out[start:end] = best_s
        t_out[start:end] = best_t
    return s_out, t_out


In [ ]:
#| export
@torch.no_grad()
def generate_samples(model, n_samples, dim, device='cpu',
                     n_steps=20, step_fn=rk4_step, warp_s=0.5, source_df=None,
                     source_scales=None, level_dims=None):
    """Sample from the flow model: integrate noise → embedding space."""
    y = sample_source((n_samples, dim), device=device, source_df=source_df,
                      source_scales=source_scales, level_dims=level_dims)
    ts = torch.linspace(0, 1, n_steps + 1)
    ts = warp_time(ts, s=warp_s)
    model.eval()
    for i in range(n_steps):
        dt = (ts[i+1] - ts[i]).item()
        y  = step_fn(model, y, ts[i].item(), dt)
    return y

In [ ]:
#| export
def mmd_rbf(x, y, n_sub=2000):
    """Unbiased MMD² with RBF kernel, median bandwidth heuristic.
    x, y: (N, D) tensors. Subsamples to n_sub for speed."""
    if x.size(0) > n_sub: x = x[torch.randperm(x.size(0))[:n_sub]]
    if y.size(0) > n_sub: y = y[torch.randperm(y.size(0))[:n_sub]]
    xy = torch.cat([x, y], dim=0)
    sigma2 = torch.cdist(xy, xy).median().pow(2).clamp(min=1e-6)
    def rbf(a, b): return torch.exp(-torch.cdist(a, b).pow(2) / (2 * sigma2))
    return (rbf(x, x).mean() + rbf(y, y).mean() - 2 * rbf(x, y).mean()).item()


In [ ]:

#| export
def wasserstein_score(x, y, n_projections=200, n_sub=2000):
    """Sliced Wasserstein distance: average 1-D Wasserstein over random projections.
    Falls back gracefully if geomloss is unavailable.
    Returns nan on numerical failure (overflow, diverged samples, etc.).
    x, y: (N, D) numpy arrays."""
    try:
        import geomloss
        loss = geomloss.SamplesLoss("sinkhorn", p=2, blur=0.05)
        xt = torch.tensor(x[:n_sub]).float()
        yt = torch.tensor(y[:n_sub]).float()
        return loss(xt, yt).item()
    except ImportError:
        pass
    except Exception:
        return float('nan')
    try:
        from scipy.stats import wasserstein_distance
        rng = np.random.default_rng(0)
        D = x.shape[1]
        projs = rng.standard_normal((D, n_projections))
        projs /= np.linalg.norm(projs, axis=0, keepdims=True)
        px, py = x[:n_sub] @ projs, y[:n_sub] @ projs
        return float(np.mean([wasserstein_distance(px[:, i], py[:, i]) for i in range(n_projections)]))
    except Exception:
        return float('nan')


In [ ]:

#| export
@torch.no_grad()
def eval_flow(model, real_embeddings, n_samples=10000, n_steps=20, warp_s=0.5, device='cpu',
              source_df=None, source_scales=None, level_dims=None):
    """Compare distributional statistics of real vs generated embeddings, per level.
    Returns flat dict with keys like 'L0/mmd', 'L0/wasserstein', 'L0/real_std', etc.
    Also returns global 'mmd' and 'wasserstein' for backward compatibility.
    real_embeddings: (N, D) tensor.
    """
    from scipy.stats import skew, kurtosis
    dim = real_embeddings.shape[1]
    idx = torch.randperm(real_embeddings.size(0))[:n_samples]
    real = real_embeddings[idx].float()
    gen  = generate_samples(model, n_samples, dim, device=device,
                            n_steps=n_steps, warp_s=warp_s, source_df=source_df,
                            source_scales=source_scales, level_dims=level_dims).cpu()
    r, g = real.numpy(), gen.numpy()

    metrics = {}
    # Global stats
    metrics['real_mean']  = float(r.mean())
    metrics['real_std']   = float(r.std())
    metrics['real_skew']  = float(skew(r.ravel()))
    metrics['real_kurt']  = float(kurtosis(r.ravel()))
    metrics['gen_mean']   = float(g.mean())
    metrics['gen_std']    = float(g.std())
    metrics['gen_skew']   = float(skew(g.ravel()))
    metrics['gen_kurt']   = float(kurtosis(g.ravel()))
    metrics['mmd']        = mmd_rbf(real, gen)
    metrics['wasserstein'] = wasserstein_score(r, g)

    # Per-level stats
    if level_dims is not None:
        offset = 0
        for i, d in enumerate(level_dims):
            rl = r[:, offset:offset+d]
            gl = g[:, offset:offset+d]
            rt, gt = torch.tensor(rl), torch.tensor(gl)
            metrics[f'L{i}/real_std']    = float(rl.std())
            metrics[f'L{i}/gen_std']     = float(gl.std())
            metrics[f'L{i}/real_kurt']   = float(kurtosis(rl.ravel()))
            metrics[f'L{i}/gen_kurt']    = float(kurtosis(gl.ravel()))
            metrics[f'L{i}/mmd']         = mmd_rbf(rt, gt)
            metrics[f'L{i}/wasserstein'] = wasserstein_score(rl, gl)
            offset += d

    w = max(len(k) for k in metrics)
    for k, v in metrics.items():
        print(f'  {k:{w}s} = {v:.4f}')
    return metrics


In [ ]:

#| export
@torch.no_grad()
def plot_level_histograms(model, real_embeddings, level_dims, n_samples=10000,
                          n_steps=20, warp_s=0.5, device='cpu', n_bins=100, source_df=None,
                          source_scales=None, epoch=None):
    """Return dict of per-level histogram figures {'L0': fig, 'L1': fig, ...}.
    level_dims: list of ints, flattened PCA dims per level e.g. [20, 80, 320, 1280]
    real_embeddings: (N, sum(level_dims)) tensor
    """
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    dim = real_embeddings.shape[1]
    idx = torch.randperm(real_embeddings.size(0))[:n_samples]
    real = real_embeddings[idx].float().numpy()
    gen  = generate_samples(model, n_samples, dim, device=device,
                            n_steps=n_steps, warp_s=warp_s, source_df=source_df,
                            source_scales=source_scales, level_dims=level_dims).cpu().numpy()
    figs = {}
    offset = 0
    for i, d in enumerate(level_dims):
        r = real[:, offset:offset+d].ravel()
        g = gen[:,  offset:offset+d].ravel()
        lim = np.percentile(np.abs(np.concatenate([r, g])), 99)
        bins = np.linspace(-lim, lim, n_bins + 1)
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.hist(r, bins=bins, alpha=0.5, color='steelblue', label='real', density=True)
        ax.hist(g, bins=bins, alpha=0.5, color='darkorange', label='gen',  density=True)
        title = f'L{i} ({d}d)'
        if epoch is not None: title += f' — Epoch {epoch}'
        ax.set_title(title)
        ax.set_xlabel('value')
        ax.legend(fontsize=8)
        plt.tight_layout()
        figs[f'L{i}'] = fig
        offset += d
    return figs

In [ ]:
#| export
@torch.no_grad()
def plot_level_scatter(model, real_embeddings, level_dims, n_samples=5000,
                       n_steps=20, warp_s=0.5, device='cpu',
                       source_df=None, source_scales=None, epoch=None):
    """Return dict of per-level 3D PCA scatter plots {'L0/real': fig, 'L0/gen': fig, ...}.
    Subsamples to n_samples, runs pca_project per level, returns plotly figures.
    """
    from midi_rae.viz import pca_project, plot_embeddings_3d
    dim = real_embeddings.shape[1]
    idx = torch.randperm(real_embeddings.size(0))[:n_samples]
    real = real_embeddings[idx].float()
    gen  = generate_samples(model, n_samples, dim, device=device,
                            n_steps=n_steps, warp_s=warp_s, source_df=source_df,
                            source_scales=source_scales, level_dims=level_dims).cpu()
    figs = {}
    offset = 0
    for i, d in enumerate(level_dims):
        r = real[:, offset:offset+d]
        g = gen[:,  offset:offset+d]
        title_sfx = f' — Epoch {epoch}' if epoch is not None else ''
        r3 = pca_project(r)
        g3 = pca_project(g)
        if r3 is not None: figs[f'L{i}/real'] = plot_embeddings_3d(r3, color_by='random', title=f'L{i} ({d}d) real{title_sfx}')
        if g3 is not None: figs[f'L{i}/gen']  = plot_embeddings_3d(g3, color_by='random', title=f'L{i} ({d}d) gen{title_sfx}')
        offset += d
    return figs

In [ ]:
#| export
def train_flow(model, dataset, n_epochs=100, lr=3e-4, batch_size=2048,
               warp_s=0.5, device='cpu', checkpoint_dir=None, save_every=10,
               eval_every=10, viz_every=50, use_wandb=False, steps_per_epoch=None,
               source_df=None, source_scales=None, checkpoint=None, cfg=None,
               lr_restart_epochs=500, grad_clip=1.0,
               repair_every=1, n_repair_projections=1, repair_chunk_size=None,
               ema_eta=0.97, ema_start_epoch=100):
    """Train flow matching model on embedding dataset.

    Source: N(0,I) sampled fresh each step.
    Target: embeddings from dataset.
    Loss:   MSE between predicted and true (constant) velocity.
    checkpoint: path to a saved checkpoint to resume from (optional).
    cfg: config dict/object passed to save_checkpoint (optional).
    lr_restart_epochs: T_0 for CosineAnnealingWarmRestarts (T_mult=2, so periods double).
    viz_every: how often (epochs) to log histograms + scatter plots to W&B.
    grad_clip: max norm for gradient clipping (0 = disabled).
    repair_every: re-pair source/target every N batches via ann_repair (0 = disabled).
    n_repair_projections: random projections per ann_repair call (more = better, slower).
    repair_chunk_size: chunk size for ann_repair (None = full batch).
    ema_eta: EMA decay rate (updated every batch).
    ema_start_epoch: epoch at which eval/sampling switches to EMA model.
    """
    from midi_rae.utils import save_checkpoint, load_checkpoint, EMAModel
    self_cond = getattr(model, 'self_condition', False)
    model = model.to(device)
    ema_model = EMAModel(model, eta=ema_eta, update_every=1, dtype=torch.float32)
    dl = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                    num_workers=2, pin_memory=(device != 'cpu'), drop_last=True)
    dl_iter = None
    _steps = steps_per_epoch or len(dl)
    if use_wandb:
        import wandb
        wandb.config.update(dict(n_epochs=n_epochs, lr=lr, batch_size=batch_size,
                                 warp_s=warp_s, dim=dataset.embeddings.shape[1],
                                 self_condition=self_cond), allow_val_change=True)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    global_step = 0
    epoch_start = 0
    real_scatter_logged = False

    if checkpoint:
        model, ckpt = load_checkpoint(model, checkpoint, return_all=True)
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        epoch_start = ckpt['epoch']
        global_step = epoch_start * _steps
        print(f"Resumed from {checkpoint} (epoch {epoch_start})")
        ema_model.ema.load_state_dict(model.state_dict())

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=lr_restart_epochs, T_mult=2, eta_min=1e-6,
        last_epoch=epoch_start - 1 if epoch_start > 0 else -1)

    for epoch in range(epoch_start, n_epochs):
        model.train()
        epoch_loss = 0.
        if steps_per_epoch:
            if dl_iter is None:
                import itertools; dl_iter = itertools.cycle(dl)
            batches = (next(dl_iter) for _ in range(_steps))
        else:
            batches = dl
        pbar = tqdm(batches, total=_steps, desc=f'Epoch {epoch+1}/{n_epochs}', leave=False)
        for target in pbar:
            target = target.to(device)
            B, D   = target.shape
            source = sample_source((B, D), device=device, source_df=source_df,
                                   source_scales=source_scales,
                                   level_dims=getattr(dataset, 'level_dims', None))
            if repair_every and global_step % repair_every == 0:
                source, target = ann_repair(source, target,
                                            n_projections=n_repair_projections,
                                            chunk_size=repair_chunk_size)

            t = torch.rand(B, 1, device=device)
            if warp_s != 1.0:
                t = warp_time(t, s=warp_s)

            x_t = (1 - t) * source + t * target
            v   = target - source

            # Self-conditioning: 50% of batches, do a first pass to get x̂₁,
            # then use it as conditioning for the actual training pass.
            x_self_cond = None
            if self_cond and torch.rand(1).item() < 0.5:
                with torch.no_grad():
                    v_first = model(x_t, t, None)
                    x_self_cond = (x_t + (1 - t) * v_first).detach()

            optimizer.zero_grad()
            v_pred = model(x_t, t, x_self_cond)
            loss   = loss_fn(v_pred, v)
            loss.backward()
            if grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
            optimizer.step()
            ema_model.update(model)

            epoch_loss += loss.item()
            pbar.set_postfix(loss=f'{loss.item():.4f}')
            global_step += 1

        scheduler.step()
        avg_loss = epoch_loss / len(dl)
        cur_lr = scheduler.get_last_lr()[0]
        print(f'Epoch {epoch+1}/{n_epochs}  loss={avg_loss:.4f}  lr={cur_lr:.2e}')
        if use_wandb: wandb.log({'train/epoch_loss': avg_loss, 'train/lr': cur_lr, 'epoch': epoch+1}, step=global_step)

        if eval_every and (epoch + 1) % eval_every == 0:
            print(f'  --- eval epoch {epoch+1} ---')
            eval_model = ema_model.ema if (epoch + 1) >= ema_start_epoch else model
            metrics = eval_flow(eval_model, dataset.embeddings, device=device, warp_s=warp_s,
                               source_df=source_df, source_scales=source_scales,
                               level_dims=getattr(dataset, 'level_dims', None))
            if use_wandb:
                import wandb
                log_dict = {f'eval/{k}': v for k, v in metrics.items()}
                if hasattr(dataset, 'level_dims') and viz_every and (epoch + 1) % viz_every == 0:
                    figs = plot_level_histograms(eval_model, dataset.embeddings,
                                               dataset.level_dims, device=device, warp_s=warp_s,
                                               source_df=source_df, source_scales=source_scales,
                                               epoch=epoch+1)
                    import matplotlib.pyplot as plt
                    for lname, fig in figs.items():
                        log_dict[f'media/hist_{lname}'] = wandb.Image(fig, caption=f'Epoch {epoch+1}')
                        plt.close(fig)
                    del figs
                    scatters = plot_level_scatter(eval_model, dataset.embeddings,
                                                 dataset.level_dims, device=device, warp_s=warp_s,
                                                 source_df=source_df, source_scales=source_scales,
                                                 epoch=epoch+1)
                    for lname, fig in scatters.items():
                        is_real = lname.endswith('/real')
                        if is_real and real_scatter_logged:
                            fig.data = []
                            continue
                        log_dict[f'media/scatter_{lname.replace("/", "_")}'] = wandb.Html(fig.to_html())
                    real_scatter_logged = True
                    del scatters
                    gc.collect()
                log_dict['epoch'] = epoch+1
                wandb.log(log_dict, step=global_step)
            model.train()

        save_checkpoint([model, ema_model.ema], epoch+1, avg_loss, cfg or {}, optimizer=optimizer,
                        save_every=save_every, tag=f'flow_{model.__class__.__name__}')

    print(f"FINISHED. Best metric: final loss={avg_loss:.4f}")
    return model


In [ ]:
#| export
#| eval: false
import hydra
from omegaconf import DictConfig

@hydra.main(version_base=None, config_path="../configs", config_name="config_swin")
def train_flow_main(cfg: DictConfig):
    import glob as _glob
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"device = {device}")

    paths = sorted(_glob.glob(os.path.expandvars(os.path.expanduser(cfg.flow.embedding_glob))))
    assert paths, f"No files found matching {cfg.flow.embedding_glob}"
    print(f"Loading {len(paths)} file(s)...")
    source_scales = list(cfg.flow.source_scales) if cfg.flow.get("source_scales") else None
    raw_df = cfg.flow.get("source_df", None)
    source_df = list(raw_df) if hasattr(raw_df, '__iter__') else raw_df
    n_levels = len(source_scales) if source_scales else None
    levels = [f'L{i}' for i in range(n_levels)] if n_levels else None
    if source_df: source_df = source_df[:n_levels]
    dataset = EmbeddingDataset(paths, levels=levels)
    dim = dataset.embeddings.shape[1]
    print(f"  {len(dataset)} samples, dim={dim}, level_dims={dataset.level_dims}")

    self_condition = cfg.flow.get('self_condition', False)
    t_dim = cfg.flow.get('t_dim', 64)
    model_type = cfg.flow.get('model_type', 'per_level')
    if model_type == 'cross_level':
        model = CrossLevelFlowModel(level_dims=dataset.level_dims,
                                    h_dim=cfg.flow.h_dim, n_layers=cfg.flow.n_layers,
                                    n_attn_layers=cfg.flow.get('n_attn_layers', 2),
                                    n_heads=cfg.flow.get('n_heads', 8),
                                    self_condition=self_condition, t_dim=t_dim)
    else:
        model = PerLevelFlowModel(level_dims=dataset.level_dims,
                                  h_dim=cfg.flow.h_dim, n_layers=cfg.flow.n_layers,
                                  self_condition=self_condition, t_dim=t_dim)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  {model.__class__.__name__}: {n_params:,} parameters (self_condition={self_condition}, t_dim={t_dim})")

    use_wandb = hasattr(cfg, "wandb") and hasattr(cfg.wandb, "flow_project")
    if use_wandb:
        import wandb
        wandb.init(project=cfg.wandb.flow_project, config=dict(cfg.flow))
        wandb.define_metric("epoch")
        wandb.define_metric("*", step_metric="epoch")
        if hasattr(cfg, "tag"): wandb.run.name = f"{cfg.tag}_{wandb.run.name}"

    checkpoint = os.path.expandvars(os.path.expanduser(cfg.get('checkpoint', '') or '')) or None

    train_flow(model, dataset,
               n_epochs=cfg.flow.n_epochs, lr=cfg.flow.lr, batch_size=cfg.flow.batch_size,
               warp_s=cfg.flow.warp_s, device=device,
               checkpoint_dir=cfg.flow.checkpoint_dir, save_every=cfg.flow.save_every,
               eval_every=cfg.flow.eval_every, viz_every=cfg.flow.get('viz_every', 50),
               use_wandb=use_wandb, steps_per_epoch=cfg.flow.steps_per_epoch,
               source_df=source_df, source_scales=source_scales,
               checkpoint=checkpoint, cfg=cfg,
               lr_restart_epochs=cfg.flow.get('lr_restart_epochs', 500),
               repair_every=cfg.flow.get('repair_every', 1),
               n_repair_projections=cfg.flow.get('n_repair_projections', 1),
               repair_chunk_size=cfg.flow.get('repair_chunk_size', None),
               ema_eta=cfg.flow.get('ema_eta', 0.97),
               ema_start_epoch=cfg.flow.get('ema_start_epoch', 100))

    if use_wandb: wandb.finish()

if __name__ == "__main__":
    train_flow_main()


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()